In [1]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [3]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

## Rutas CSV

In [4]:
TOPIC_TO_CLASSIFY = "geometría"

In [5]:
if TOPIC_TO_CLASSIFY:
    topic = TOPIC_TO_CLASSIFY.lower().strip().replace(" ", "_")
    CSV_TO_MERGE = root / f"csv/clasificados_tema/ejercicios_clasificados_{topic}.csv"
else:
    print("¡NO SE HA INDICADO TEMA A COMBINAR!")
CSV_CLASSIFIED = root / "csv/ejercicios_clasificados.csv"

Función para modificar el id de ejercicios multitipo

In [6]:
def modify_id(row):

    id = row["id"]
    tipo_ejercicio = row["tipo_ejercicio"]

    # get position of year
    year_position = id.find("20")

    # insert tipo_ejercicio into id
    modified_tipo_ejercicio = remove_accents(tipo_ejercicio
                                             .lower()
                                             .replace(" ", "_")
                                             )
    for word in ["_en", "_de", "_y"]:
        modified_tipo_ejercicio = modified_tipo_ejercicio.replace(word, "")

    new_id = id[:year_position] + modified_tipo_ejercicio + "_" + id[year_position:]

    return new_id 

## Combinar enunciados con clasificaciones

In [7]:
def combine_exercises(csv_to_merge, csv_enun = CSV_CLASSIFIED): 
    df_enun = pd.read_csv(csv_enun, encoding="utf-8")
    
    # set index id
    df_enun = df_enun.set_index("id")

    # load classified exercises to be merged
    df_to_merge = pd.read_csv(root / "csv" / csv_to_merge)
    
    # set index id
    df_to_merge = df_to_merge.set_index("id")

    # merge classified (df_classified) with already classified exercises (df_enun)
    df_enun.update(df_to_merge.loc[:, ["tipo_ejercicio", "observaciones"]])

    # restore id to be a column again
    df_enun = df_enun.reset_index(drop = False)  

    # rearrange columns
    rearranged_cols = ['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
                   'tipo_ejercicio','observaciones', 'enunciado',
                   "id",
                   'info_ejercicio', 'archivo', 'imagen'
                   ]
    df_enun = df_enun[rearranged_cols]

    # check for multitype exercises and explode them
    df_enun["tipo_ejercicio"] = df_enun["tipo_ejercicio"].str.split("|")
    df_enun = df_enun.explode("tipo_ejercicio", ignore_index=True)

    # find the indexes of multitype exercises
    idx_multitype = df_enun[df_enun.duplicated(subset=["id"], keep=False)].index

    # modify their id
    new_ids = df_enun.loc[idx_multitype].apply(modify_id, axis=1)
    df_enun.loc[idx_multitype, "id"] = new_ids

    # check if there are still duplicated id values
    if not any(df_enun.duplicated(subset=["id"], keep=False)):
    
        # save merged df_enun 
        df_enun.to_csv(CSV_CLASSIFIED, index = False, encoding="utf-8")

    return df_enun                          

In [8]:
# df_to_merge = pd.read_csv(CSV_TO_MERGE)

In [9]:
# df_enun = pd.read_csv(CSV_CLASSIFIED)

In [10]:
# df_enun.loc[df_enun.tema == "geometría", "observaciones"] = pd.NA

In [11]:
# df_enun.to_csv(CSV_CLASSIFIED, index=False, encoding="utf-8")

In [12]:
# df_to_merge.sample()

In [13]:
f = combine_exercises(CSV_TO_MERGE)

In [14]:
f[f.tema == TOPIC_TO_CLASSIFY].sample(5)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,observaciones,enunciado,id,info_ejercicio,archivo,imagen
14195,Mates II,2018,Reserva 4,"Ejercicio 4, opción B",geometría,planos,NaN,a) Determina la ecuación del plano que pasa po...,mates_ii_geometria_vectores_planos_angulos_pla...,MATEMÁTICAS II. 2018. RESERVA 4. EJERCICIO 4. ...,source_pdf_files/Mates II/geometria/2018 - Geo...,enunciados_ejercicios/Mates II/geometria/png/p...
13656,Mates II,2018,Reserva 1,"Ejercicio 4, opción A",geometría,planos,NaN,Se considera el plano  de ecuación x2yz 6....,mates_ii_geometria_posicion_relativa_planos_po...,MATEMÁTICAS II. 2018. RESERVA 1. EJERCICIO 4. ...,source_pdf_files/Mates II/geometria/2018 - Geo...,enunciados_ejercicios/Mates II/geometria/png/p...
7083,Mates II,2022,Reserva 1,Ejercicio 7,geometría,posición relativa,NaN,x 0 x y1\nConsidera las rectas r   y s...,mates_ii_geometria_posicion_relativa_planos_pl...,MATEMÁTICAS II. 2022. RESERVA 1. EJERCICIO 7.,source_pdf_files/Mates II/geometria/2022 - Geo...,enunciados_ejercicios/Mates II/geometria/png/p...
17953,Mates II,2016,Reserva 3,"Ejercicio 4, opción B",geometría,planos,NaN,"Sea r la recta que pasa por los puntos A(1,1,0...",mates_ii_geometria_posicion_relativa_recta_pos...,MATEMÁTICAS II. 2016. RESERVA 3. EJERCICIO 4. ...,source_pdf_files/Mates II/geometria/2016 - Geo...,enunciados_ejercicios/Mates II/geometria/png/p...
16222,Mates II,2017,Reserva 3,"Ejercicio 4, opción B",geometría,recta,NaN,"Considera los puntos A(1,1,1) , B(0,2,2) , C(...",mates_ii_geometria_recta_planos_posicion_relat...,MATEMÁTICAS II. 2017. RESERVA 3. EJERCICIO 4. ...,source_pdf_files/Mates II/geometria/2017 - Geo...,enunciados_ejercicios/Mates II/geometria/png/p...


In [15]:
f[(f.tema == TOPIC_TO_CLASSIFY) & (f.tipo_ejercicio.isna())]

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,observaciones,enunciado,id,info_ejercicio,archivo,imagen


Guardar ejercicios clasificados

In [17]:
f.to_csv(root / "csv/ejercicios_clasificados.csv", index = False, encoding="utf-8")